In [5]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt #matplotlib绘图库
import copy #python的copy模块提供了浅拷贝和深拷贝操作
import os
import sys

### 读取并渲染点云图
第一步总是先读取点云图，才能继续其他操作

In [13]:
print("Load a ply point cloud, print it, and render it")
pcd = o3d.io.read_point_cloud(r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\fragment.ply")
print(pcd)
print(np.asarray(pcd.points))
#注意这里，不仅仅是可视化，要裁剪cropped.json的话是draw_geometries_with_editing，不仅仅是draw_geometries
o3d.visualization.draw_geometries_with_editing([pcd])

Load a ply point cloud, print it, and render it
PointCloud with 196133 points.
[[0.65234375 0.84686458 2.37890625]
 [0.65234375 0.83984375 2.38430572]
 [0.66737998 0.83984375 2.37890625]
 ...
 [2.00839925 2.39453125 1.88671875]
 [2.00390625 2.39488506 1.88671875]
 [2.00390625 2.39453125 1.88793314]]


### 手动裁剪点云，得到ply文件
可视化用的是draw_geometries_with_editing([pcd])  
运行以后会打开open3D的窗口。你可以在窗口里通过交互操作选取一个polygon区域；教程里常见的操作流程是先调整视角，然后按`K`锁定视角进入选择模式，用`Ctrl + 左键`依次选择多边形顶点，最后用`S`保存选择区域。保存出来的就是类似 `cropped.ply` 的文件。

cropped.json就是要裁剪的多边形的形状，需要手动写json文件，和这里的不一样。   

### crop point cloud 点云裁剪 ROI
读取点云 -> 定义裁剪区域vol -> crop -> 得到ROI

In [14]:
print("Load a polygon volume and use it to crop the original point cloud")  #加载一个多边形区域，并使用它来裁剪原始点云
#cropped.json保存了一个事先选好的多边形空间区域，里面包含了裁剪区域的顶点坐标
#把这个区域读取并保存到变量vol中
vol = o3d.visualization.read_selection_polygon_volume(
    r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\cropped.json")
#用这个区域裁剪出椅子，将裁剪后的点云保存到变量chair
chair = vol.crop_point_cloud(pcd)
o3d.visualization.draw_geometries([chair])

Load a polygon volume and use it to crop the original point cloud
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: 不支持请求的转换操作。 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: 不支持请求的转换操作。 


### Point cloud distance 点云距离 可利用距离做裁剪
Open3D provides the method`compute_point_cloud_distance`to compute the distance from a source point cloud to a target point cloud. I.e., it computes for each point in the source point cloud the distance to the closest point in the target point cloud.  
In the example below we use the function to compute the difference between two point clouds. Note that this method could also be used to compute the Chamfer distance between two point clouds.

Open3D 提供了`compute_point_cloud_distance`方法，用于计算源点云到目标点云的距离。  
也就是说，对于源点云中的每一个点，它都会计算该点到目标点云中最近点的距离。
在下面的示例中，我们使用这个函数来计算两个点云之间的差异。需要注意的是，该方法也可以用于计算两个点云之间的 Chamfer Distance（倒角距离）。

In [ ]:
#裁剪出椅子
pcd = o3d.io.read_point_cloud(r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\fragment.ply")
vol = o3d.visualization.read_selection_polygon_volume(
    r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\cropped.json")
chair = vol.crop_point_cloud(pcd)

#用原点云pcd和裁剪出的椅子chair做对比计算点云距离
#source点云：pcd，target点云：chair
dists = pcd.compute_point_cloud_distance(chair)
dists = np.asarray(dists)  #转换成numpy数组
ind = np.where(dists > 0.01)[0] #找出完整点云pcd中所有与裁剪点云chair距离大于0.01的点的索引，即删掉椅子以及椅子附近的点
#ind = np.where(dists < 0.01)[0] #仅保留椅子，删掉其他离椅子远的点 #但是也没必要了，因为上面已经用了json提取出椅子了
pcd_without_chair = pcd.select_by_index(ind)  #根据索引提取点
o3d.visualization.draw_geometries([pcd_without_chair])  #可视化裁剪掉椅子后的点云

### Bounding volumes 包围体
The PointCloud geometry type has bounding volumes as all other geometry types in Open3D. Currently, Open3D implements an `AxisAlignedBoundingBox` and an `OrientedBoundingBox` that can also be used to crop the geometry.
  
两种包围盒：`AxisAlignedBoundingBox`轴对齐包围盒（AABB）以及`OrientedBoundingBox`有向包围盒 / 定向包围盒（OBB）  
这两种包围盒也可以用来对几何体（例如点云）进行裁剪（crop）。  

AABB 的特点是：包围盒的三个方向永远和全局坐标系的 X、Y、Z 轴平行。所以即使椅子本身是倾斜的，这个盒子也不会跟着椅子旋转
OBB 可以旋转，它会根据点云本身的空间分布寻找一个合适的方向，因此通常能比 AABB 更紧密地包住物体。

简单来说，就是先用一个三维盒子把你感兴趣的点云区域框起来，然后只保留盒子里面的点。这其实就很适合你之前说的点云预处理中的 ROI 裁剪。

In [18]:
aabb = chair.get_axis_aligned_bounding_box()
aabb.color = (1,0,0) #红色
obb = chair.get_oriented_bounding_box()
obb.color = (0,1,0) #绿色
o3d.visualization.draw_geometries([chair, aabb, obb])

### Convex hull  凸包 适合描述已经获得的点云的外部形状、边界、体积等
**可能没啥用，就没看**

The convex hull of a point cloud is the smallest convex set that contains all points. Open3D contains the method `compute_convex_hull` that computes the convex hull of a point cloud. The implementation is based on`Qhull`.
In the example code below we first sample a point cloud from a mesh and compute the convex hull that is returned as a triangle mesh. Then, we visualize the convex hull as a red `LineSet`.

点云的凸包是包含所有点的最小凸集。Open3D 提供了`compute_convex_hull`方法，用于计算点云的凸包。该功能的实现基于`Qhull`。
在下面的示例代码中，我们首先从一个网格中采样得到点云，然后计算该点云的凸包。计算得到的凸包以三角网格的形式返回。
之后，我们将这个凸包以红色的 `LineSet`（线集合）的形式进行可视化